# Overview

This notebook is intended to showcase some analytics that can be run on EV Charger time series data, with a focus on sumary statistics. It is not meant to be comprehensive, many further analyses could be envisioned.

The notebook is structured as follows:
- A Setup section for establishing a connection and performing basic data checks
- A Data Retrieval section for actual retrieval of the time series for EV Chargers as well as Meters
- Three Analysis sections:
    1. Comparing total load of EV Chargers with that of Meters over the entire period that data is available for
    2. Summarizing EV Charger load over the period by day and by month
    3. Visualizing temporal patterns in EV Charger load, e.g. versus day of the week, hour of the day or month of the year

These kinds of analyses can help utilities better understand the nature of EV Charger load, which in turn can lead to better planning and management of EV charging infrastructure or EV Charger rate design.

For more details about EV Charger usage analysis and how it can be done using Awesense's platform, please refer to the [UC09-01 - EV Charging and Use of Reserved Capacity Analysis.pdf](https://github.com/Awesense/edm-app-examples/blob/master/use_cases/usecase_descriptions/UC09-01%20-%20EV%20Charging%20and%20Use%20of%20Reserved%20Capacity%20Analysis.pdf) document.

# Setup

In [1]:
import getpass
import urllib.parse
import pandas as pd
import plotly.express as px
import numpy as np

## Conection

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

get_ipython().run_line_magic('load_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://$edm_name:$edm_password@$edm_address/edm')
get_ipython().run_line_magic('config', 'SqlMagic.displaycon = False')
get_ipython().run_line_magic('config', 'SqlMagic.feedback = False')

# Delete the credential variables for security purposes.

del edm_name, edm_password


EDM login information


## Data checks

#### Checking the awefice grid is present

In [3]:
%%sql

SELECT * FROM grid;

grid_id,description,last_updated
awefice,awefice,2021-01-01 20:00:00+00:00
North Central Zone,North Central Zone,2021-01-01 20:00:00+00:00


#### Looking up the EV Chargers in this grid.

In [4]:
%%sql

SELECT *
FROM grid_element_data_source geds, grid_element ge
WHERE ge.grid_element_id = geds.grid_element_id
AND ge.grid_id = 'awefice'
AND ge.type = 'EVCharger';

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction,grid_id_1,grid_element_id_1,type_1,customer_type,phases_1,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
a3dfe4ea-faaa-4a1c-a43e-b111d52e4684,awefice,evch_10,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_10,None,awefice,evch_10,EVCharger,ev_charger,ABC,None,True,True,False,None,con_135_t2,VCN.evch_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': None, 'label': 'evch_10', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_10', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'ChargePoint Home Flex [CPH50-NEMA14-50-L23]', 'rating_kva': '', 'max_current': 32, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-30', 'active_charging_power': 7680, 'parent_transformer_id': 'transformer_36', 'active_generation_power': 0}"
ab23affd-cc3d-43aa-9db7-13078ca38b14,awefice,evch_12,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_12,None,awefice,evch_12,EVCharger,ev_charger,ABC,None,True,True,False,None,con_152_t2,VCN.evch_12,both,None,0101000020E610000032415CF294C65EC0F99D2EC423A24840,"{'name': None, 'label': 'evch_12', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_12', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'Tesla Supercharger [Tesla]', 'rating_kva': '', 'max_current': 330, 'max_voltage': 410, 'is_dc_charger': True, 'voltage_level': 'LV', 'charging_level': 'level_3', 'connector_type': 'TESLA', 'active_charging_power': 135000, 'parent_transformer_id': 'transformer_47', 'active_generation_power': 0}"
d581fcac-42f1-4ecb-9a11-34735c71bf2f,awefice,evch_20,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_20,None,awefice,evch_20,EVCharger,ev_charger,ABC,None,True,True,False,None,con_228_t2,VCN.evch_20,both,None,0101000020E6100000B0AF7695AAC65EC0DC65CD8A58A24840,"{'name': None, 'label': 'evch_20', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_20', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'Grizzl-E EV Charger [GR1-14-24-P]', 'rating_kva': '', 'max_current': 16, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-30', 'active_charging_power': 3840, 'parent_transformer_id': 'transformer_63', 'active_generation_power': 0}"
bf63eabb-1b4a-4937-addc-77853cbc9171,awefice,evch_24,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_24,None,awefice,evch_24,EVCharger,ev_charger,ABC,None,True,True,False,None,con_312_t2,VCN.evch_24,both,None,0101000020E61000001C613FDF81C65EC0E40C592457A24840,"{'name': None, 'label': 'evch_24', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_24', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'ChargePoint CPE 200 [CPE200T-S-CHD-CMB]', 'rating_kva': '', 'max_current': 125, 'max_voltage': 500, 'is_dc_charger': True, 'voltage_level': 'LV', 'charging_level': 'level_3', 'connector_type': '[CHAdeMO,CCS1]', 'active_charging_power': 50000, 'parent_transformer_id': 'transformer_92', 'active_generation_power': 0}"
bc9a2651-736b-44e2-b3fd-e04463803a09,awefice,evch_3,"[2021-01-01 08:00:00+00:00, None)",A,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_3,None,awefice,evch_3,EVCharger,ev_charger,A,None,True,True,False,None,con_40_t2,VCN.evch_3,both,None,0101000020E61000005BCDA4FDBEC65EC0E0B44C793CA24840,"{'name': None, 'label': 'evch_3', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_3', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'Tesla Gen 3 Wall Connector [1457768-**-*]', 'rating_kva': '', 'max_current': 16, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_l

# Data Retrieval

In [5]:
# Defining Data Retrieval Function.

def get_hourly_kwh(grid_id, grid_element_type, new_col_name_for_id):
    """
    Retrieves the consumption (kWh) time series for a given grid_element_type from a given grid, aggregated per hour,
        converted from UTC to Vancouver timezone, and returns the final output as a dataframe.
        
    Note that to avoid infinitesimally small load which impacts the capacity analysis a lot, 
        the values are rounded to 2 decimal places.
    """
    
    result =  %sql SELECT geds.grid_element_id, \
                        date_trunc('hour', timestamp) at time zone 'America/Vancouver' as date_hour, \
                        round(sum(value)::numeric, 2) AS "kWh", \
                        geds.type \
                    FROM grid_element ge \
                    JOIN grid_element_data_source geds \
                        ON ge.grid_id = geds.grid_id \
                            AND ge.grid_element_id = geds.grid_element_id \
                    JOIN ts_data_source_select(geds.grid_element_data_source_id, 'kWh', null) tdss \
                        ON true \
                    WHERE ge.grid_id = '{grid_id}' \
                        AND ge.type = '{grid_element_type}' \
                    GROUP BY geds.grid_element_id, date_hour, geds.type; 

    df = result.DataFrame()
    df['kWh'] = df['kWh'].astype(float)
    df.rename(columns={'grid_element_id': new_col_name_for_id}, inplace=True)
    return df

## Retrieving and summarizing EV Charger time series

In [6]:
# Get time series consumption data for EVChargers, aggregated to hourly granularity.
df_evch = get_hourly_kwh('awefice', 'EVCharger', 'evch_id')

In [7]:
df_evch.head()

,evch_id,date_hour,kWh,type
0,evch_10,2021-04-27 00:00:00,0.0,TRUE_CONSUMPTION
1,evch_10,2022-11-18 04:00:00,0.0,TRUE_CONSUMPTION
2,evch_10,2022-11-24 11:00:00,0.0,TRUE_CONSUMPTION
3,evch_10,2021-04-10 17:00:00,0.0,TRUE_CONSUMPTION
4,evch_10,2022-05-22 08:00:00,0.0,TRUE_CONSUMPTION


In [8]:
# Summary stats for EV Chargers.
df_evch.describe()

,date_hour,kWh
count,175177,175177.000000
mean,2022-12-31 23:30:36.364362752,2.516342
min,2021-01-01 00:00:00,0.000000
25%,2021-12-31 23:00:00,0.000000
50%,2022-12-31 23:00:00,0.000000
75%,2023-12-31 23:00:00,0.000000
max,2024-12-31 23:00:00,73.210000
std,NaN,9.205144


## Retrieving and summarizing the Meter time series

In [9]:
# Get time series consumption data for Meters, aggregated to hourly granularity.
df_meter = get_hourly_kwh('awefice', 'Meter', 'meter_id')

In [10]:
df_meter.head()

,meter_id,date_hour,kWh,type
0,m_4,2022-09-16 17:00:00,0.63,CONSUMER
1,m_4,2022-06-06 01:00:00,0.39,CONSUMER
2,m_4,2022-06-02 13:00:00,0.00,CONSUMER
3,m_4,2022-01-19 14:00:00,2.55,CONSUMER
4,m_4,2021-06-17 17:00:00,0.32,CONSUMER


In [11]:
# Summary stats for Meters.
df_meter.describe()

,date_hour,kWh
count,1156781,1.156781e+06
mean,2023-01-01 08:47:41.869618944,2.137538e+00
min,2021-01-01 00:00:00,0.000000e+00
25%,2022-01-01 04:00:00,4.500000e-01
50%,2023-01-01 08:00:00,1.020000e+00
75%,2024-01-01 13:00:00,2.180000e+00
max,2024-12-31 23:00:00,7.936000e+01
std,NaN,4.899079e+00


## Further data munging (reshaping, joining Meters and EV Chargers)

In [12]:
# Pivot the Meter data to separate out Producer (flow in) and Consumer (flow out) components.
df_meter_split = df_meter.pivot(index=['meter_id', 'date_hour'], columns='type', values='kWh').rename_axis(None, axis=1).reset_index()
df_meter_split.head()

,meter_id,date_hour,CONSUMER,PRODUCER
0,m_1,2021-01-01 00:00:00,1.92,NaN
1,m_1,2021-01-01 01:00:00,1.84,NaN
2,m_1,2021-01-01 02:00:00,1.87,NaN
3,m_1,2021-01-01 03:00:00,1.77,NaN
4,m_1,2021-01-01 04:00:00,1.96,NaN


In [13]:
# Summary row counts
df_meter_split.groupby(['meter_id']).count()

,date_hour,CONSUMER,PRODUCER
meter_id,,,
m_1,35060,35060,0
m_10,35015,35015,35015
m_11,35060,35060,35060
m_12,35018,35018,0
m_13,35060,35060,0
m_14,35060,35060,0
m_15,35041,35041,0
m_16,35060,35060,0
m_17,35060,35060,0


In [14]:
# Replace Nan values with 0.
df_meter_split.replace(np.nan, 0, inplace=True)
df_meter_split.head()

,meter_id,date_hour,CONSUMER,PRODUCER
0,m_1,2021-01-01 00:00:00,1.92,0.0
1,m_1,2021-01-01 01:00:00,1.84,0.0
2,m_1,2021-01-01 02:00:00,1.87,0.0
3,m_1,2021-01-01 03:00:00,1.77,0.0
4,m_1,2021-01-01 04:00:00,1.96,0.0


In [15]:
# Compute the net of flow in and out for each meter.
df_meter_split['NET'] = df_meter_split['CONSUMER'] - df_meter_split['PRODUCER']
df_meter_split.head()

,meter_id,date_hour,CONSUMER,PRODUCER,NET
0,m_1,2021-01-01 00:00:00,1.92,0.0,1.92
1,m_1,2021-01-01 01:00:00,1.84,0.0,1.84
2,m_1,2021-01-01 02:00:00,1.87,0.0,1.87
3,m_1,2021-01-01 03:00:00,1.77,0.0,1.77
4,m_1,2021-01-01 04:00:00,1.96,0.0,1.96


In [16]:
# Summary stats of Meter data.
df_meter_split.describe()

,date_hour,CONSUMER,PRODUCER,NET
count,946466,946466.000000,946466.000000,946466.000000
mean,2023-01-01 08:57:28.684263680,2.355732,0.256790,2.098941
min,2021-01-01 00:00:00,0.000000,0.000000,-46.400000
25%,2022-01-01 04:00:00,0.660000,0.000000,0.660000
50%,2023-01-01 09:00:00,1.400000,0.000000,1.400000
75%,2024-01-01 13:00:00,2.310000,0.000000,2.310000
max,2024-12-31 23:00:00,79.360000,46.400000,79.360000
std,NaN,4.854922,2.394356,5.523863


#### Retrieve the EV Charger to Meter correspondence from the DB:

In [17]:
%%sql evchs_meter <<

SELECT ge1.grid_element_id as evch_id,
    ge2.grid_element_id as meter_id
FROM grid_element ge1
JOIN grid_get_sources('awefice', ge1.grid_element_id, True) ge2 ON True
WHERE ge1.type='EVCharger'
    AND ge2.type = 'Meter';

Returning data to local variable evchs_meter


In [18]:
# Convert to data frame for later use.
evchs_meter = evchs_meter.DataFrame()
evchs_meter

,evch_id,meter_id
0,evch_10,m_10
1,evch_12,m_12
2,evch_20,m_20
3,evch_24,m_24
4,evch_3,m_3


In [19]:
# Add-in the information about the corresponding Meter in the EV Charger time series data frame.
df_evch_m = pd.merge(df_evch, evchs_meter, on='evch_id', how='inner')
df_evch_m.head()

,evch_id,date_hour,kWh,type,meter_id
0,evch_10,2021-04-27 00:00:00,0.0,TRUE_CONSUMPTION,m_10
1,evch_10,2022-11-18 04:00:00,0.0,TRUE_CONSUMPTION,m_10
2,evch_10,2022-11-24 11:00:00,0.0,TRUE_CONSUMPTION,m_10
3,evch_10,2021-04-10 17:00:00,0.0,TRUE_CONSUMPTION,m_10
4,evch_10,2022-05-22 08:00:00,0.0,TRUE_CONSUMPTION,m_10


# Analysis: Meter-EVCharger Correspondence

## Summing up data over all time

In [20]:
# Print out the time ranges of the data for both Meters and EV Chargers for informational purposes
print(df_meter['date_hour'].min())
print(df_meter['date_hour'].max())
print(df_evch['date_hour'].min())
print(df_evch['date_hour'].max())

2021-01-01 00:00:00
2024-12-31 23:00:00
2021-01-01 00:00:00
2024-12-31 23:00:00


In [21]:
# Compute the sum of EV Charger consumption from all the EV Chargers corresponding to each Meter.
meter_evch_kwh_portion = df_evch_m.groupby('meter_id').sum('kWh').reset_index()
meter_evch_kwh_portion

,meter_id,kWh
0,m_10,46180.61
1,m_12,275368.25
2,m_20,13820.04
3,m_24,91702.00
4,m_3,13734.34


In [22]:
# Sum up the Meter time series (flow in, flow out and net) for each Meter.
meter_kwh_total_net = df_meter_split.groupby('meter_id').sum(numeric_only=True).reset_index()
meter_kwh_total_net.sort_values(by='NET', ascending=False)

,meter_id,CONSUMER,PRODUCER,NET
3,m_12,413855.85,0.00,413855.85
2,m_11,338665.10,27688.78,310976.32
1,m_10,164052.74,80.91,163971.83
18,m_26,123555.94,0.00,123555.94
20,m_3,63367.14,0.00,63367.14
6,m_15,63351.45,0.00,63351.45
12,m_20,54831.75,0.00,54831.75
17,m_25,52014.27,0.00,52014.27
4,m_13,50959.97,0.00,50959.97
5,m_14,49197.46,0.00,49197.46


In [23]:
# Add in the information about respective EV Charger consumption for each Meter
meter_combined_data = pd.merge(meter_evch_kwh_portion, meter_kwh_total_net, on='meter_id', how='outer')
meter_combined_data = meter_combined_data.rename(
    columns={'kWh': 'EV Charger Consumption'
             , 'PRODUCER': 'Meter Flow In'
             , 'CONSUMER': 'Meter Flow Out'
             , 'NET': 'Meter Net'})
meter_combined_data

,meter_id,EV Charger Consumption,Meter Flow Out,Meter Flow In,Meter Net
0,m_10,46180.61,164052.74,80.91,163971.83
1,m_12,275368.25,413855.85,0.00,413855.85
2,m_20,13820.04,54831.75,0.00,54831.75
3,m_24,91702.00,163562.09,201969.82,-38407.73
4,m_3,13734.34,63367.14,0.00,63367.14
5,m_1,NaN,41543.00,0.00,41543.00
6,m_11,NaN,338665.10,27688.78,310976.32
7,m_13,NaN,50959.97,0.00,50959.97
8,m_14,NaN,49197.46,0.00,49197.46
9,m_15,NaN,63351.45,0.00,63351.45


## Plotting

In [24]:
# Plot per-Meter EVCharger consumption alongside Meter's Net flow, sorted by the latter descending. 
fig = px.bar(meter_combined_data.sort_values(by='Meter Net', ascending=False)
       , x='meter_id', y=['Meter Net', 'EV Charger Consumption']
       , labels={'value':'kWh', 'meter_id':'Meter'}
       , barmode='group'
       , color_discrete_map={'Meter Net': 'gray', 'EV Charger Consumption': 'red'}
       , title='<b>All-time</b> EV Charger Consumption compared to Meter Net<br>Meters sorted descending by their <i>Net</i>'
      )
fig.update_xaxes(tickangle=45)
fig.show()

In [25]:
# Plot per-Meter EVCharger consumption alongside Meter's Net flow, sorted by the former descending. 
fig = px.bar(meter_combined_data.sort_values(by='EV Charger Consumption', ascending=False)
       , x='meter_id', y=['Meter Net', 'EV Charger Consumption']
       , labels={'value':'kWh', 'meter_id':'Meter'}
       , barmode='group'
       , color_discrete_map={'Meter Net': 'gray', 'EV Charger Consumption': 'red'}
       , title='<b>All-time</b> EV Charger Consumption compared to Meter Net<br>Meters sorted descending by their <i>downstream EV Charger Consumption</i>'
      )
fig.update_xaxes(tickangle=45)
fig.show()

In [26]:
# Plot per-Meter EVCharger consumption alongside Meter's Flow In&Out, sorted by the former descending. 
fig = px.bar(meter_combined_data.sort_values(by='EV Charger Consumption', ascending=False)
       , x='meter_id', y=['Meter Flow Out', 'Meter Flow In', 'EV Charger Consumption']
       , labels={'value':'kWh', 'meter_id':'Meter'}
       , barmode='group'
       , color_discrete_map={'Meter Flow Out': 'blue','Meter Flow In': 'lightblue', 'EV Charger Consumption': 'red'}
       , title='<b>All-time</b> EV Charger Consumption compared to Meter Flow In/Out<br>Meters sorted descending by their <i>downstream EV Charger Consumption</i>'
      )
fig.update_xaxes(tickangle=45)
fig.show()

# Analysis: EV Charger Consumption Over All Time

## Helper functions

In [27]:
def sum_loads(df, groupby_list, sum_list):
    """
    Returns a dataframe that is grouped by `groupby_list` columns and summed `sum_list` columns.
    Note that the `groupby_list` and `sum_list` may be a single string.
    """
    
    df_final = df.groupby(groupby_list
                     ).sum(sum_list, min_count=1).reset_index()  # min_count=1 keeps null as null in summation.
    
    return df_final

In [28]:
def add_date_range(df, time_col, desc_str):
    """
    Returns a string with a date range added to `desc_str` string, to be used in a chart's title.
    """

    # Pull out min and max dates to use in the title.
    min_date = min(df[time_col].dt.date).strftime('%b %d, %Y')
    max_date = max(df[time_col].dt.date).strftime('%b %d, %Y')

    # Prepare the title.
    title = desc_str + " (" + min_date + " ~ " + max_date + ")"
    
    return title

## Daily EV Charger consumption over entire history

In [29]:
# Create a copy to prepare the data with.
df = df_evch.copy()

# Pull out a date component out of timestamp to aggregate over.
df['date'] = df['date_hour'].dt.date

# Sum all EV Chargers loads for each date.
df_daily = sum_loads(df, 'date', 'kWh')

# Prepare a title with a date range.
title = add_date_range(df, time_col='date_hour', 
                       desc_str='<b>Daily</b> Sum of All EV Charger Load')

# Plot.
fig = px.line(df_daily, 
        x = 'date', y = 'kWh',
        title = title,
        labels = {'date':'Date'},
        width = 1000, height = 500,
        range_y = [0, max(df_daily['kWh'])*1.05]
      )
fig.update_xaxes(tickangle=45)
fig.show()

## Monthly EV Charger consumption over entire history

In [30]:
# Create a copy to prepare the data with.
df = df_evch.copy()

# Pull out a date component out of timestamp to aggregate over.
df['year_month'] = df['date_hour'].dt.strftime('%Y-%m')

# Sum EV Chargers for all parkades for each month.
df_monthly = sum_loads(df, 'year_month', 'kWh')

# Order the data by correct year month notion and not by alphabetically.
df_monthly = df_monthly.sort_values('year_month').reset_index()

# Prepare a title with a date range.
title = add_date_range(df, time_col = 'date_hour', 
                       desc_str = '<b>Monthly</b> Sum of All EV Charger Load')

# Plot.
fig = px.bar(df_monthly, 
        x = 'year_month', y = 'kWh',
        title = title,
        labels = {'year_month':'Date'},
        width = 1000, height = 500,
        range_y = [0, max(df_monthly['kWh'])*1.05]
      )
fig.update_xaxes(tickangle=45)
fig.show()

# Analysis: EV Charger consumption patterns w.r.t. various units of time

## Data processing and generic helper functions

In [31]:
# Helper function
def enrich_data_for_plotting(df_evch):
    """
    Adds columns to the given dataframe for various units of time (e.g. hour-of-day, day-of-week, month)
    by computing them from the date_hour column of the given dataframe.
    """
    
    # Create a copy of the raw dataframe to work with.
    df_all = df_evch.copy()

    # Pull out year-week combinations to separate out each day based on the week.
    df_all.loc[:,'year_week'] = df_all['date_hour'].dt.strftime('%Y-%W')

    # Assign word version of days of the week.
    df_all.loc[:,'day'] = df_all['date_hour'].dt.day_name()    

    # Assign numeric days of the week to maintain its ordinality.
    df_all.loc[:,'day_n'] = df_all['date_hour'].dt.weekday
    
    # Pull out the hours.
    df_all.loc[:,'hour'] = df_all['date_hour'].dt.hour

    # Pull out the months.
    df_all.loc[:,'month'] = df_all['date_hour'].dt.month_name()

    # Assign numeric month of year to maintain its ordinality.
    df_all.loc[:,'month_n'] = df_all['date_hour'].dt.month

    return df_all

In [32]:
# Enrich EV Charger data with hour-of-day/day-of-week/month information.
df_evch_enh = enrich_data_for_plotting(df_evch)
df_evch_enh.head()

,evch_id,date_hour,kWh,type,year_week,day,day_n,hour,month,month_n
0,evch_10,2021-04-27 00:00:00,0.0,TRUE_CONSUMPTION,2021-17,Tuesday,1,0,April,4
1,evch_10,2022-11-18 04:00:00,0.0,TRUE_CONSUMPTION,2022-46,Friday,4,4,November,11
2,evch_10,2022-11-24 11:00:00,0.0,TRUE_CONSUMPTION,2022-47,Thursday,3,11,November,11
3,evch_10,2021-04-10 17:00:00,0.0,TRUE_CONSUMPTION,2021-14,Saturday,5,17,April,4
4,evch_10,2022-05-22 08:00:00,0.0,TRUE_CONSUMPTION,2022-20,Sunday,6,8,May,5


In [33]:
# Helper function
def prep_data_for_overlaid_line_chart(df_evch_enh_agg, column, line_length):
    """
    Process data for overlaid line chart using the given column of the given dataframe 
    and breaking the data into segments of line_length length (e.g. 7 for day-of-week chart, 
    24 for hour-of-day chart).
    The easiest way to obtain a chart of overlaid lines is to insert rows with Nones into 
    the dataframe at the places where a new segment is supposed to start.
    """
    df = df_evch_enh_agg.copy()

    # Order properly to then be able to remove first&last partial weeks.
    df.sort_values(['year_week', 'evch_id', column], ascending = True, inplace = True, ignore_index=True)

    # find portion of first week (if it doesn't start on Monday)
    first_0 = next(idx for idx, val in enumerate(df[column], 1) if val == 0) - 1

    # find portion of last week (if it doesn't end on Sunday)
    last_6 = len(df) - next(idx for idx, val in enumerate(df.loc[::-1, column], 1) if val == 6)

    # remove first and last partial weeks
    df = df.iloc[first_0 : (last_6+1),].reset_index(drop=True)

    # Re-sort by evch_id first
    df.sort_values(['evch_id', 'year_week', column], ascending = True, inplace = True, ignore_index=True)

    # Get the distinct evch_ids.
    evch_ids = df['evch_id'].unique()
    evch_ids_and_counts = df.groupby('evch_id')['kWh'].count().reset_index().sort_values('evch_id', ascending=True)

    # Insert Nones with proper evch_id for plotting purposes (to break into segments)
    idx = 0
    for i in range(len(evch_ids_and_counts)):
        evch_id_val, count = evch_ids_and_counts.iloc[i,:]
        n_weeks = count // line_length
        break_rows = pd.DataFrame({"hour": np.NaN, "kWh": None, "evch_id": evch_id_val, "year_week": None}
                                  , index=[idx + line_length * (i + 1) - 0.5 for i in range(n_weeks)])
        #df = df.append(break_rows, ignore_index=False)
        df = pd.concat([df, break_rows], ignore_index=False)
        idx = idx + count
    # Get the Nones to be in the right spots
    df = df.sort_index().reset_index(drop=True)

    return df

## Day-of-Week Analysis

In [34]:
# Helper function
def plot_evch_daily_load_vs_dow(df_all, evch_id, chart_type):
    """
    Makes a plot of EV Charger Load vs. Day of Week.
    Can plot one EV Charger or many (if evch_id='all'), in either multi-line form or boxplot form 
    as specified by chart_type.
    """

    # Prepare a title with an appropriate date range.
    df_for_title = (df_all if evch_id == 'all' else df_all[df_all['evch_id'] == evch_id])
    title = add_date_range(df_for_title,
                           time_col = 'date_hour', 
                           desc_str = 'EV Charger <b>Daily</b> Sum of Load vs. <b>Day-of-week</b>' + '<br>' + ('<b>All</b> EV Chargers' if evch_id == 'all' else 'EV Charger: <b>' + evch_id + '</b>')
                          )

    # Sum EV Load data to days.
    df_agg = sum_loads(df_all, ['evch_id', 'year_week', 'day', 'day_n', 'month', 'month_n'], 'kWh')

    # Filter out for a single evch id if needed.
    if evch_id == 'all':
        df = df_agg
    else:
        df = df_agg[df_agg['evch_id'] == evch_id].copy()
    
    # Plot appropriate chart based on the chart type specified.
    if chart_type=='box':
        # Order by numeric days of the week to order the plot properly.
        df.sort_values('day_n', ascending = True, inplace = True)
        # Plot.
        fig = px.box(df, x='day', y='kWh',
                         title=title,
                         labels={'day':'Day of Week'},
                         width=1000, height=500)
        fig.update_traces(marker_color = 'red')

    elif chart_type=='line':
        # Split into weekly chunks each with 7 daily values
        df = prep_data_for_overlaid_line_chart(df, 'day_n', 7)
        
        # Plot.
        fig = px.line(df, x='day', y='kWh', 
                      color = 'evch_id',
                      title=title,
                      labels={'day':'Day of Week', 'evch_id':'EV Charger ID'}, 
                      width=1000, height=500)
        # Set the multiple lines to be more transparent
        fig.update_traces(opacity=0.2)

    fig.show()

In [37]:
# Plotting day-of-week boxplots and line charts, first for all EV Chargers combined and then one EV Charger per plot.
plot_evch_daily_load_vs_dow(df_evch_enh, evch_id='all', chart_type='line')
plot_evch_daily_load_vs_dow(df_evch_enh, evch_id='all', chart_type='box')
# Below loop is currently commented out because Jupyter has issues rendering that many plots.
# for evch_id in df_evch['evch_id'].unique():
#     plot_evch_daily_load_vs_dow(df_evch_enh, evch_id=evch_id, chart_type='line')
#     plot_evch_daily_load_vs_dow(df_evch_enh, evch_id=evch_id, chart_type='box')

/var/folders/vp/7ptlp5l934vdh1lvmpgk4qyc0000gn/T/ipykernel_85673/2989829630.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/vp/7ptlp5l934vdh1lvmpgk4qyc0000gn/T/ipykernel_85673/2989829630.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/vp/7ptlp5l934vdh1lvmpgk4qyc0000gn/T/ipykernel_85673/2989829630.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or a

## Hour-of-Day Analysis

In [38]:
def plot_evch_hourly_load_vs_hod(df_all, evch_id, chart_type):
    """
    Makes a plot of EV Charger Load vs. Hour-of-Day.
    Can plot one EV Charger or many, in either multi-line form or boxplot form
    """

    # Prepare a title with an appropriate date range.
    df_for_title = (df_all if evch_id == 'all' else df_all[df_all['evch_id'] == evch_id])
    title = add_date_range(df_for_title,
                           time_col = 'date_hour', 
                           desc_str = 'EV Charger <b>Hourly</b> Load vs. <b>Hour-of-Day</b>' + '<br>' + ('<b>All</b> EV Chargers' if evch_id == 'all' else 'EV Charger: <b>' + evch_id + '</b>')
                          )

    # Sum EV Load data to hours.
    df_agg = sum_loads(df_all, ['evch_id', 'year_week', 'hour'], 'kWh')

    # Filter out for a single evch id if needed.
    if evch_id == 'all':
        df = df_agg
    else:
        df = df_agg[df_agg['evch_id'] == evch_id].copy()
    
    # Plot appropriate chart based on the chart type specified.
    if chart_type=='box':
        # Order by numeric days of the week to order the plot properly.
        df.sort_values('hour', ascending = True, inplace = True)
        
        fig = px.box(df, x='hour', y='kWh',
                         title=title,
                         labels={'hour':'Hour of Day'},
                         width=1000, height=500)
        fig.update_traces(marker_color = 'red')

    elif chart_type=='line':
        # Split into daily chunks each with 24 hourly values
        df = prep_data_for_overlaid_line_chart(df, 'hour', 24)
        
        # Plot.
        fig = px.line(df, x='hour', y='kWh', 
                      color = 'evch_id',
                      title=title,
                      labels={'hour':'Hour of Day', 'evch_id':'EV Charger ID'}, 
                      width=1000, height=500)
        # Set the multiple lines to be more transparent
        fig.update_traces(opacity=0.2)

    fig.show()

In [40]:
# Plotting hour-of-day boxplots and line charts, first for all EV Chargers combined and then one EV Charger per plot.
plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id='all', chart_type='line')
plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id='all', chart_type='box')
# Below loop is currently commented out because Jupyter has issues rendering that many plots.
# for evch_id in df_evch['evch_id'].unique():
#     plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id=evch_id, chart_type='line')
#     plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id=evch_id, chart_type='box')

/var/folders/vp/7ptlp5l934vdh1lvmpgk4qyc0000gn/T/ipykernel_85673/2989829630.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/vp/7ptlp5l934vdh1lvmpgk4qyc0000gn/T/ipykernel_85673/2989829630.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/vp/7ptlp5l934vdh1lvmpgk4qyc0000gn/T/ipykernel_85673/2989829630.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or a

## Month-of-Year Analysis

In [41]:
def plot_evch_daily_load_vs_moy(df_all, evch_id):
    """
    Makes a plot of Daily EV Charger Load vs. Month of Year.
    Can plot one EV Charger or many, but in boxplot form only
    """

    # Prepare a title with an appropriate date range.
    df_for_title = (df_all if evch_id == 'all' else df_all[df_all['evch_id'] == evch_id])
    title = add_date_range(df_for_title,
                           time_col = 'date_hour', 
                           desc_str = 'EV Charger <b>Daily</b> Sum of Load vs. <b>Month-of-Year</b>' + '<br>' + ('<b>All</b> EV Chargers' if evch_id == 'all' else 'EV Charger: <b>' + evch_id + '</b>')
                          )

    # Sum EV Load data to days.
    df_agg = sum_loads(df_all, ['evch_id', 'year_week', 'month', 'month_n'], 'kWh')

    # Filter out for a single evch id if needed.
    if evch_id == 'all':
        df = df_agg
    else:
        df = df_agg[df_agg['evch_id'] == evch_id].copy()

    # Order by numeric days of the week to order the plot properly.
    df.sort_values('month_n', ascending = True, inplace = True)
    
    # Plot.
    fig = px.box(df, x='month', y='kWh',
                     title=title,
                     labels={'month':'Month'},
                     width=1000, height=500)
    fig.update_traces(marker_color = 'red')

    fig.show(config={'staticPlot': True})

In [42]:
# plot_evch_daily_load_vs_moy(df_evch_enh, evch_id='all')
for evch_id in df_evch['evch_id'].unique():
    plot_evch_daily_load_vs_moy(df_evch_enh, evch_id=evch_id)